# 07 — Alerta Telegram (Fase 0)

**Objetivo:** Enviar mensaje (y foto opcional) al bot de Telegram (REQ-04).


## Prerrequisitos

`TELEGRAM_BOT_TOKEN`, `TELEGRAM_CHAT_ID` en `.env`; eventos de **04**.


## 1. Setup


In [1]:
from __future__ import annotations

from pathlib import Path

import requests

from _common.io import (
    ensure_scripts_on_path,
    env_or_none,
    load_dotenv_repo,
    read_jsonl,
    setup_logging,
    stage_output_dir,
)
from loguru import logger

ensure_scripts_on_path()
load_dotenv_repo()
setup_logging()


## 2. Configuration


In [2]:
EVENTS_PATH = stage_output_dir("04_events") / "events.jsonl"
SEMANTIC_DIR = stage_output_dir("05_semantic")
TRACK_VIDEO = stage_output_dir("03_track") / "annotated.mp4"
THUMB = stage_output_dir("01_capture") / "frame_000000.jpg"

TOKEN = env_or_none("TELEGRAM_BOT_TOKEN")
CHAT_ID = env_or_none("TELEGRAM_CHAT_ID")
SKIPPED = False


## 3. Construir payload


In [3]:
events = read_jsonl(EVENTS_PATH)
if not events:
    raise RuntimeError("Sin eventos para alertar")

event = events[-1]
text = (
    f"[VisionOps] {event.get('type', 'event').upper()}\n"
    f"{event.get('message', '')}\n"
    f"Severity: {event.get('severity')}\n"
    f"Zone: {event.get('zone')}"
)
logger.info("Mensaje: {}", text[:120])


21:59:26 | INFO | Mensaje: [VisionOps] WARNING
Persona 4 en zona ROI
Severity: medium
Zone: roi_placeholder


## 4. Enviar a Telegram


In [4]:
response_summary = {"status": "pending"}

if not TOKEN or not CHAT_ID:
    SKIPPED = True
    response_summary = {
        "status": "SKIPPED",
        "reason": "TELEGRAM_BOT_TOKEN o TELEGRAM_CHAT_ID ausentes",
    }
    logger.warning(response_summary["reason"])
else:
    base = f"https://api.telegram.org/bot{TOKEN}"
    r = requests.post(
        f"{base}/sendMessage",
        json={"chat_id": CHAT_ID, "text": text},
        timeout=30,
    )
    response_summary = {"status": "ok" if r.ok else "error", "http": r.status_code, "body": r.text[:500]}
    if THUMB.is_file() and r.ok:
        with THUMB.open("rb") as photo:
            rp = requests.post(
                f"{base}/sendPhoto",
                data={"chat_id": CHAT_ID, "caption": event.get("type", "")},
                files={"photo": photo},
                timeout=60,
            )
        response_summary["photo_http"] = rp.status_code

response_summary


21:59:26 | WARNING | TELEGRAM_BOT_TOKEN o TELEGRAM_CHAT_ID ausentes


{'status': 'SKIPPED',
 'reason': 'TELEGRAM_BOT_TOKEN o TELEGRAM_CHAT_ID ausentes'}

## 5. Validación


In [5]:
assert response_summary["status"] in ("ok", "SKIPPED", "error")
print(f"Telegram: {response_summary['status']}")


Telegram: SKIPPED
